# imports

In [1]:
from preprocess import get_data
from model_utils import export_model_data_json, get_auc_roc
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from tabicl import TabICLClassifier 
import torch

c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-302\sos-routier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
new_df = get_data()
new_df

,VITESSE_MA,METEO_beau,METEO_chute de neige,METEO_couvert,METEO_grêle,METEO_pluie,METEO_pluie givrante,ETAT_ROUTE_enneigée,ETAT_ROUTE_humide,ETAT_ROUTE_mouillé,ETAT_ROUTE_neige fondante,ETAT_ROUTE_route enneigée,ETAT_ROUTE_sèche,ETAT_ROUTE_verglacée,HEURE_SIN,HEURE_COS,VICTIME
0,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.866,-0.500,False
1,80.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000,-1.000,False
2,50.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.707,0.707,False
3,50.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.000,0.000,False
4,80.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.866,-0.500,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18586,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18587,120.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.707,0.707,False
18588,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18589,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.500,-0.866,False


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.2,
    random_state=42,
)

In [4]:
def print_cross_validation_results(regressor, X_train, y_train):
    scoring = ["accuracy", "f1", "roc_auc"]    
    cv_results = cross_validate(
        regressor,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=1,
        verbose=0,
        return_train_score=True,
    )
    print("Train Accuracy: {:.2f}".format(cv_results["train_accuracy"].mean()))
    print("Validation Accuracy: {:.2f}".format(cv_results["test_accuracy"].mean()))
    print("Train F1-Score: {:.2f}".format(cv_results["train_f1"].mean()))
    print("Validation F1-Score: {:.2f}".format(cv_results["test_f1"].mean()))
    print("Train AUC-ROC: {:.2f}".format(cv_results["train_roc_auc"].mean()))
    print("Validation AUC-ROC: {:.2f}".format(cv_results["test_roc_auc"].mean()))

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.2,
    random_state=42,
)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"


In [6]:
tfm = TabICLClassifier(device=device)

tfm.fit(X_train, y_train)
print_cross_validation_results(tfm, X_train, y_train)

c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-302\sos-routier\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-302\sos-routier\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-302\sos-routier\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-30

Train Accuracy: 0.59
Validation Accuracy: 0.58
Train F1-Score: 0.48
Validation F1-Score: 0.50
Train AUC-ROC: 0.63
Validation AUC-ROC: 0.63


In [7]:
y_proba_test = tfm.predict_proba(X_test)[:, 1]

auc, roc = get_auc_roc(y_test, y_proba_test)

export_model_data_json("tabularFoundation", auc, roc)